# 회귀분석 03. 정규화

## 학습 목표
- 핵심 정의와 정리를 자신의 말로 설명한다.
- 기본 개념 예제를 손계산으로 확인한다.
- Python 코드와 시각화를 통해 직관을 검산한다.

## 핵심 개념
- 손실함수
- SVD/PCA
- 분류
- 정규화

## 기본 개념 예제
가중치 크기에 벌점을 주면 모델이 어떻게 단순해지는지 본다.

## 실제 응용 예제
고차원 특성에서 과적합을 줄인다.

## 이론 정리

### 정의와 관점
- 데이터분석은 자료의 구조, 변동성, 이상치, 관계를 해석 가능한 형태로 요약한다.
- 머신러닝은 손실함수를 최소화해 데이터에서 예측 규칙을 학습한다.
- PCA는 데이터 분산이 큰 직교 방향을 찾아 차원을 줄인다.
- 정규화는 모델 복잡도에 벌점을 주어 과적합을 줄이는 방법이다.

### 핵심 명제와 정리
- 최소제곱과 로지스틱 회귀는 선형대수와 최적화의 결합으로 이해할 수 있다.
- SVD는 PCA, 저차원 근사, 노이즈 제거의 공통 핵심 도구다.
- 경사하강법은 손실함수의 기울기를 따라 파라미터를 반복 갱신한다.
- 일반화 성능은 훈련오차보다 검증오차와 데이터 분포 변화에 의해 평가된다.

### 계산과 학습 절차
- 데이터 품질, 결측, 이상치, 단위, 시간 순서를 먼저 점검한다.
- 분석은 EDA, 모형 설정, 학습, 검증, 해석, 전달 순서로 진행한다.
- 모델 결과는 계수, 잔차, 혼동행렬, 예측구간 등 문제에 맞는 진단과 함께 본다.
- 차원축소는 정보 손실과 해석 가능성을 함께 고려한다.

### 자주 생기는 오해
- 높은 정확도는 데이터 누수나 편향된 검증 방식에서 나올 수 있다.
- 상관관계 기반 모델은 인과 효과를 자동으로 말해주지 않는다.
- 시계열 데이터는 무작위 섞기 전에 시간 순서와 미래 정보 누수를 확인해야 한다.

### 증명으로 연결하기
- 학습 알고리즘 분석은 손실함수의 미분, 볼록성, 일반화 오차, 편향-분산 분해를 사용한다.
- PCA의 최적성은 분산 최대화와 재구성오차 최소화가 같은 문제임을 보이는 데서 나온다.
- 회귀 추론은 오차 가정 아래 추정량의 분포와 표준오차 계산으로 이어진다.

### 이 챕터에서 꼭 확인할 질문
- 기본 개념 예제 "가중치 크기에 벌점을 주면 모델이 어떻게 단순해지는지 본다."에서 실제로 사용한 정의는 무엇인가?
- 응용 예제 "고차원 특성에서 과적합을 줄인다."에서 어떤 가정이 현실을 단순화하고 있는가?
- 코드가 연속 대상을 이산화한다면, 격자나 표본 수를 바꾸어도 결론이 유지되는가?
- 손계산 가능한 작은 사례와 노트북 결과가 같은 결론을 주는가?

## 0. 실행 준비

아래 셀은 프로젝트 루트의 `common/math_viz.py`를 찾아서 현재 챕터의 출력 폴더를 자동으로 설정합니다. Jupyter Lab을 프로젝트 루트에서 열면 가장 안정적으로 동작합니다.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display


CHAPTER_RELATIVE_DIR = Path("4학년_심화_과목과_연구_주제/09_회귀분석/ch03_정규화")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "common" / "math_viz.py").exists():
            return candidate
    raise RuntimeError("common/math_viz.py를 찾지 못했습니다. Jupyter Lab을 프로젝트 루트에서 열어 주세요.")


ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DIR = ROOT / CHAPTER_RELATIVE_DIR
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
sys.path.insert(0, str(ROOT / "common"))

from math_viz import PROFILES, run_profile

PROFILE = "machine_learning_math"
TITLE = "회귀분석 - 정규화"
CONCEPT_EXAMPLE = "가중치 크기에 벌점을 주면 모델이 어떻게 단순해지는지 본다."
APPLICATION_EXAMPLE = "고차원 특성에서 과적합을 줄인다."

print("project root:", ROOT)
print("chapter dir:", NOTEBOOK_DIR)
print("profile:", PROFILE)

## 1. 이번 챕터의 시각화 코드 읽기

먼저 실제로 실행될 함수를 확인합니다. 코드를 읽으면서 입력값, 이산화 방식, 그래프가 의미하는 수학적 대상을 표시해 보세요.

In [ ]:
import inspect

print(inspect.getsource(PROFILES[PROFILE]))

## 2. 실행하고 결과 확인하기

아래 셀을 실행하면 `outputs/visualization.png`가 생성되고, 노트북 안에도 바로 표시됩니다.

In [ ]:
run_profile(
    profile=PROFILE,
    title=TITLE,
    concept=CONCEPT_EXAMPLE,
    application=APPLICATION_EXAMPLE,
    output_dir=OUTPUT_DIR,
)

display(Image(filename=str(OUTPUT_DIR / "visualization.png")))

## 3. 변형 실험

- 표본 수, 격자 크기, 초기값, 학습률, 경계조건 중 하나를 바꿔 보세요.
- 그림이 안정적으로 유지되는 범위와 결론이 바뀌는 범위를 나누어 적어 보세요.
- 손계산 가능한 작은 예제를 만들어 코드 결과와 비교해 보세요.

In [ ]:
# 여기에 자신만의 변형 실험을 작성하세요.
# 예: common/math_viz.py에서 위에 출력된 함수의 파라미터를 복사해 와서
#     표본 수, 구간, 초기값 등을 바꾼 뒤 다시 그려 볼 수 있습니다.
